# Ablation — is the gain DIVERSITY, or is it DATA VOLUME?

The main experiment compared A\* demonstrations (12,819 pairs) against BCO-GA
demonstrations (51,003 pairs). Because the population planner rolls K = 3
variants per scenario it supplies **3.98x more data**, so the headline 4.54x
unique-state advantage confounds diversity with volume. Subsampling both sets
to a common budget leaves only **1.26x** genuine per-sample diversity.

These runs test whether that 1.26x is enough to reproduce the performance gap.

## Design

Every arm is capped to the **same demonstration budget** (`CAP`), set to the
size of the smallest set (K=1, 11,173 pairs). Volume is then held constant and
K is the only thing that varies, so a single sweep answers both questions:

| arm | source | what it tests |
|---|---|---|
| `warm_astar_cap` | `demos_astar.npz` | deterministic teacher at the common budget |
| `warm_bco_k1_cap` | `demos_bco_k1.npz` | population teacher, least diverse |
| `warm_bco_k3_cap` | `demos_bco.npz` | population teacher, K = 3 (the paper's setting) |
| `warm_bco_k5_cap` | `demos_bco_k5.npz` | population teacher, most diverse |

**How to read the outcome**
* `k3_cap` still beats `astar_cap` → diversity is the mechanism; the claim survives
* `k3_cap` collapses to `astar_cap` → the gain was data volume; the claim must change
* performance rising monotonically k1 → k3 → k5 → direct, dose-response evidence
  for diversity, because K changes nothing else

`cold`, `warm_astar` and `warm_bco` from the main experiment are **not** re-run;
they remain the uncapped reference. Run this notebook once per `SEED` (1, 2, 3).

In [ ]:
from google.colab import files
up = files.upload()   # marl_env.py, train_marl.py, demos_astar.npz,
                      # demos_bco_k1.npz, demos_bco.npz, demos_bco_k5.npz
print(sorted(up))

In [ ]:
SEED  = 1        # <-- set to 1, then 2, then 3 (re-run the whole notebook each time)
STEPS = 500000
CAP   = 11173    # common demonstration budget = size of the smallest set (K=1)

import numpy as np
for f in ['demos_astar.npz', 'demos_bco_k1.npz', 'demos_bco.npz', 'demos_bco_k5.npz']:
    n = len(np.load(f)['actions'])
    assert n >= CAP, f'{f} has only {n} pairs, below the {CAP} budget'
    print(f'{f:22s} {n:7d} pairs  -> capped to {CAP}')
print(f'\nseed={SEED}  steps={STEPS}  common budget={CAP} pairs')

In [ ]:
# Arm 1/4 - deterministic teacher at the common budget.
# The [BC] log reports pairs and unique states AFTER capping - record both.
!python train_marl.py --tag warm_astar_cap --warmstart demos_astar.npz --demo-cap {CAP} --total-steps {STEPS} --seed {SEED}

In [ ]:
# Arm 2/4 - population teacher, K = 1 (least diverse)
!python train_marl.py --tag warm_bco_k1_cap --warmstart demos_bco_k1.npz --demo-cap {CAP} --total-steps {STEPS} --seed {SEED}

In [ ]:
# Arm 3/4 - population teacher, K = 3 (the setting used in the paper)
!python train_marl.py --tag warm_bco_k3_cap --warmstart demos_bco.npz --demo-cap {CAP} --total-steps {STEPS} --seed {SEED}

In [ ]:
# Arm 4/4 - population teacher, K = 5 (most diverse)
!python train_marl.py --tag warm_bco_k5_cap --warmstart demos_bco_k5.npz --demo-cap {CAP} --total-steps {STEPS} --seed {SEED}

In [ ]:
!zip -r ablation_seed{SEED}.zip runs
from google.colab import files
files.download(f'ablation_seed{SEED}.zip')
# then change SEED above and re-run the whole notebook